In [2]:
import lxml.etree as ET

In [3]:
tree = ET.parse("../xml/editio.xml")

In [10]:
ns = {
    "tei": "http://www.tei-c.org/ns/1.0",
    "xml": "http://www.w3.org/XML/1998/namespace",
}

In [11]:
def element_text(element):
    if element is None:
        return None

    return " ".join("".join(element.itertext()).split())


def extract_siglum(abbr):
    if abbr is None:
        return None

    base = abbr.text or ""

    superscript = abbr.find("./tei:hi[@rend='superscript']", ns)

    return {
        "base": base.strip(),
        "superscript": element_text(superscript),
        "display": element_text(abbr),
    }


def process_witness(witness, parent_id=None, group_id=None):
    witness_id = witness.get("{http://www.w3.org/XML/1998/namespace}id")

    abbr = witness.find("./tei:abbr[@type='siglum']", ns)
    title = witness.find("./tei:title", ns)

    record = {
        "id": witness_id,
        "siglum": extract_siglum(abbr),
        "title": element_text(title),
        "parent": parent_id,
        "group": group_id,
    }

    records.append(record)

    nested_list = witness.find("./tei:listWit", ns)

    if nested_list is not None:
        for child in nested_list.findall("./tei:witness", ns):
            process_witness(
                child,
                parent_id=witness_id,
                group_id=group_id,
            )

In [12]:
records = []

manuscripts = tree.xpath(
    "//tei:div[@xml:id='bibliography-manuscripts']/tei:listWit",
    namespaces=ns,
)

for list_wit in manuscripts:
    group_id = list_wit.get("{http://www.w3.org/XML/1998/namespace}id")

    for witness in list_wit.findall("./tei:witness", ns):
        process_witness(
            witness,
            group_id=group_id,
        )

In [18]:
import json
# print the JSON output in a pretty format
json_output = json.dumps(records, indent=4, ensure_ascii=False)

print(json_output)

[
    {
        "id": "N",
        "siglum": {
            "base": "N",
            "superscript": null,
            "display": "N"
        },
        "title": "Codex Neapolitanus V A 8",
        "parent": null,
        "group": "mss-first-family"
    },
    {
        "id": "N1",
        "siglum": {
            "base": "N",
            "superscript": "1",
            "display": "N1"
        },
        "title": null,
        "parent": "N",
        "group": "mss-first-family"
    },
    {
        "id": "N2",
        "siglum": {
            "base": "N",
            "superscript": "2",
            "display": "N2"
        },
        "title": null,
        "parent": "N",
        "group": "mss-first-family"
    },
    {
        "id": "N3",
        "siglum": {
            "base": "N",
            "superscript": "3",
            "display": "N3"
        },
        "title": null,
        "parent": "N",
        "group": "mss-first-family"
    },
    {
        "id": "G",
        "siglum": {
       

In [19]:
# Dump the JSON output to a file
with open("../json/sigla.json", "w", encoding="utf-8") as f:
    json.dump(records, f, indent=4, ensure_ascii=False)